# Waggle Dance Video Conversion and Analysis

This notebook does three things:

1. Converts raw video files, such as `.avi` or `.mts`, to `.mp4`
2. Helps check that the conversion worked
3. Loads waggle annotator CSV output and plots waggle directions

## How to use this notebook

Run the cells from top to bottom.

You only need to edit the paths in the **USER INPUT** cell below.
Do not edit the code cells unless instructed.

## Requirements

Before running this notebook, make sure the following are installed:

1. Python packages:
   - pandas
   - numpy
   - matplotlib

2. FFmpeg

FFmpeg is needed for video conversion. It is not a normal Python package. Download from:
https://ffmpeg.org/download.html

It must be installed on the computer and available from the system PATH.

To test whether FFmpeg works, this notebook includes a diagnostic cell below.

In [ ]:
# install required packages
%pip install pandas numpy matplotlib

## Imports

In [ ]:
from pathlib import Path
import subprocess
import shutil
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

In [ ]:
# ==================================
# USER INPUT
# ==================================
# Edit paths if necessary

# Folder containing the original video files
video_folder = Path(r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\20250123\a")

# Folder where converted MP4 files will be saved
output_folder = Path(r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\mp4\George\mp4\20250123")

# Video extensions that should be converted
video_extensions = [".avi", ".mts"]

In [ ]:
# ==================================
# LOCATE FOLDERS AND VIDEOS
# ==================================

# Make sure output folder exists
output_folder.mkdir(parents=True, exist_ok=True)

# Find all video files in video_folder with the extensions from your user input
video_files = []

for ext in video_extensions:
    video_files.extend(video_folder.glob(f"*{ext}"))
    video_files.extend(video_folder.glob(f"*{ext.upper()}"))

video_files = sorted(video_files)

print(f"Found {len(video_files)} video files:")
for file in video_files:
    print(file.name)

if len(video_files) == 0:
    raise ValueError("No video files found. Check video_folder and video_extensions.")

## Check that folders and FFmpeg work

Run this cell before converting videos.

If something is wrong, stop here and fix the problem before continuing.

In [ ]:
print("Python being used:")
print(sys.executable)

print("\nCurrent working directory:")
print(os.getcwd())

print("\nVideo folder exists?")
print(video_folder.exists(), video_folder)

print("\nOutput folder exists?")
print(output_folder.exists(), output_folder)

print("\nFFmpeg found?")
print(shutil.which("ffmpeg"))

In [ ]:
# ==================================
# FIND FILES INSIDE FOLDER
# ==================================

if not video_folder.exists():
    raise FileNotFoundError(f"Video folder does not exist: {video_folder}")

print("Files inside video folder:")

for file in video_folder.iterdir():
    print(file.name, "→", file.suffix)

In [ ]:
video_files = [
    file for file in video_folder.iterdir()
    if file.suffix.lower() in video_extensions
]

print("Number of matching videos:", len(video_files))

for file in video_files:
    print(file.name)

if len(video_files) == 0:
    print("\nNo videos found.")
    print("Check that your videos are directly inside the video_folder.")
    print("Also check that their file endings are included in video_extensions.")

In [ ]:
output_folder.mkdir(parents=True, exist_ok=True)

print("MP4 output folder is ready:")
print(output_folder)

## Convert all videos in folder to MP4

In [ ]:
converted = 0
failed = 0

for file in video_files:
    output_file = output_folder / (file.stem + ".mp4")

    command = [
        "ffmpeg",
        "-y",
        "-i", str(file),
        "-map", "0:v:0",
        "-vf", "bwdif=mode=send_frame:parity=tff:deint=all,fps=25",
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "15",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-an",
        str(output_file)
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode == 0 and output_file.exists():
        print(f"Converted {file.name} → {output_file.name}")
        converted += 1
    else:
        print(f"FAILED: {file.name}")
        print(result.stderr[-1000:])
        failed += 1

print("\nConversion finished.")
print("Successfully converted:", converted)
print("Failed:", failed)
print("Output folder:", output_folder)

# Run waggle dance annotator and extract angles

Follow the instructions from the official repository:

https://github.com/BioroboticsLab/bb_waggledance_annotator

Typical workflow:

1. Activate environment (this step must be run **in a terminal**, not inside the notebook)

conda activate waggle-annotator 

dance_analysis_2 -p /path/to/directory #where you have the mp4 videos stored


2. Run annotator on MP4 videos

3. Export results as **CSV file**

4. Use the CSV file in the next step of this notebook


In [ ]:
# ==================================
# LOAD ANNOTATION DATA (CSV file from annotator)
# ==================================

# CSV file produced by the waggle annotator
# Where the CSV files from the annotator is saved (change path if necessary)
annotation_file = Path(r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\dome\rightcameraangle_probably\20250121\22b_solar_2025-01-21 12-03-51_waggle_annotations.csv")

# Histogram settings
num_bins = 36

df = pd.read_csv(annotation_file)

print("Data loaded successfully")
print("Number of rows:", len(df))

df.head()

## Extract Waggle Angles

The CSV file contains waggle directions stored as lists of vectors.

This function extracts those vectors and converts them into **angles in radians**.

In [ ]:
def extract_angles(df):

    angles = []

    for entry in df["waggle_start_positions"]:

        if pd.isna(entry):
            continue

        directions = ast.literal_eval(entry)

        for x, y in directions:

            angle = np.arctan2(-y, -x)
            angles.append(angle)

    return np.array(angles)


angles = extract_angles(df)

print("Number of extracted waggle directions:", len(angles))

## Plot Waggle Directions

This creates a **polar histogram** showing the distribution of waggle dance angles.

In [ ]:
# Extract angles from CSV
all_angles = []
for entry in df["waggle_directions"]:
    if pd.isna(entry):
        continue
    directions = ast.literal_eval(entry)
    for x, y in directions:
        angle = np.arctan2(y, -x)  # your mapping
        all_angles.append(angle)

angles = np.array(all_angles)

# Create polar figure
fig, ax = plt.subplots(figsize=(6,6), subplot_kw={'projection':'polar'})

# 0° = up, clockwise rotation
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

# Dot plot on the outer circle
radii = np.ones_like(angles)  # all points at max radius
ax.scatter(angles, radii, color='blue', alpha=0.7, s=50)

# Force the radius to 1 (so dots lie exactly on the circumference)
ax.set_rlim(0, 1)

# Hide radial ticks/labels for clean look
ax.set_rticks([])
ax.set_yticklabels([])

# Optional: hide radial gridlines if you want only the circle outline
ax.grid(True)  # keep circular grid (optional)
ax.spines['polar'].set_visible(True)  # circle border

ax.set_title("Waggle Run Directions (dots on circumference)")

plt.show()

In [ ]:
# print the angles in degrees
print(np.degrees(angles[:10]))

## Statistics

In [ ]:
# mean angle
mean_angle = np.arctan2(
    np.mean(np.sin(angles)),
    np.mean(np.cos(angles))
)

print("Mean direction (degrees):", np.degrees(mean_angle))

In [ ]:
# concentration (how tight the cluster is)
R = np.sqrt(
    np.mean(np.cos(angles))**2 +
    np.mean(np.sin(angles))**2
)

print("Concentration (R):", R)

In [ ]:
# Sanity check for angles
# Define a few example (x, y) vectors like in your CSV
vectors = [
    (0.6178, 0.7863),
    (0.1249, 0.9922),
    (0.6672, 0.7449),
    (0.0, 1.0),
    (0.4596, 0.8881)
]

fig, ax = plt.subplots(figsize=(5,5), subplot_kw={'projection':'polar'})

# Plot each vector
for x, y in vectors:
    angle = np.arctan2(y, -x)  # your conversion
    r = 1  # unit circle
    ax.plot([0, angle], [0, r], marker='o', label=f"({x:.2f},{y:.2f})")

# Set up like video: 0° = up, clockwise
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_rticks([])  # hide radius ticks
ax.set_title("Sanity check: vectors to angles")
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.show()